In [1]:
import sys
import json
import joblib
from time import time
from collections import defaultdict
from itertools import product
# TO CHANGE
BASEDIR = "../../.."
sys.path.insert(0, BASEDIR)

In [2]:
from pprint import pprint

In [3]:
from src.kg_model import KnowledgeGraphModelConfig, KnowledgeGraphModel
from src.utils.data_structs import QueryInfo

from src.pipelines.qa.kg_reasoning.medium_reasoner.entities_extractor import EntitiesExtractorConfig, EntitiesExtractor
from src.pipelines.qa.kg_reasoning.medium_reasoner.entities2nodes_matching import Entities2NodesMatcherConfig, Entities2NodesMatcher

from src.pipelines.qa.kg_reasoning.weak_reasoner.knowledge_retriever import KnowledgeRetrieverConfig, KnowledgeRetriever
from src.pipelines.qa.kg_reasoning.weak_reasoner.knowledge_retriever.traversal_methods import WaterCirclesRetriever, WaterCirclesSearchConfig, \
    MixturedTripletsRetriever, MixturedGraphSearchConfig, AStarTripletsRetriever, AStarGraphSearchConfig, \
        NaiveBFSTripletsRetriever, NaiveBFSGraphSearchConfig, NaiveTripletsRetriever, NaiveGraphSearchConfig, \
            BeamSearchTripletsRetriever, GraphBeamSearchConfig

from src import PersonalAI, PersonalAIConfig
from src.pipelines.qa.kg_reasoning.weak_reasoner import WeakKGReasonerConfig
from src.pipelines.qa.kg_reasoning.medium_reasoner import MediumKGReasonerConfig
from src.pipelines.qa import QAPipelineConfig
from src.pipelines.qa.kg_reasoning import KnowledgeGraphReasonerConfig
from src.pipelines.qa.query_preprocessing import QueryPreprocessorConfig

from src.utils.logger import LogLevel, Logger

/home/dzigen/Desktop/Projects/PersonalAI/.pai_venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### Preparing testings configs

In [4]:
AVAILABLE_RETRIEVER_CONFIGS = {
    'naive_retriever': KnowledgeRetrieverConfig(
        retriever_method='naive_retriever',
        retriever_config=NaiveGraphSearchConfig(),
        filter_method=None,
        filter_config=None
    ),
    # 'astar': KnowledgeRetrieverConfig(
    #     retriever_method='astar',
    #     retriever_config=AStarGraphSearchConfig(),
    #     filter_method=None,
    #     filter_config=None
    # ),
    'beamsearch': KnowledgeRetrieverConfig(
        retriever_method='beamsearch',
        retriever_config=GraphBeamSearchConfig(),
        filter_method=None,
        filter_config=None
    ),
    'watercircles': KnowledgeRetrieverConfig(
        retriever_method='watercircles',
        retriever_config=WaterCirclesSearchConfig(),
        filter_method=None,
        filter_config=None
    ),
    # 'naive_bfs': KnowledgeRetrieverConfig(
    #     retriever_method='naive_bfs',
    #     retriever_config=NaiveBFSGraphSearchConfig(),
    #     filter_method=None,
    #     filter_config=None
    # ),
    # 'mixture (beamsearch + astart)': KnowledgeRetrieverConfig(
    #     retriever_method='mixture',
    #     retriever_config=MixturedGraphSearchConfig(
    #         retriever1_name='beamsearch',
    #         retriever1_config=GraphBeamSearchConfig(),
    #         retriever2_name='astar',
    #         retriever2_config=AStarGraphSearchConfig()
    #     ),
    #     filter_method=None,
    #     filter_config=None
    # ),
    'mixture (beamsearch + watercircles)': KnowledgeRetrieverConfig(
        retriever_method='mixture',
        retriever_config=MixturedGraphSearchConfig(
            retriever1_name='beamsearch',
            retriever1_config=GraphBeamSearchConfig(),
            retriever2_name='watercircles',
            retriever2_config=WaterCirclesSearchConfig()
        ),
        filter_method=None,
        filter_config=None
    ),
    'mixture (beamsearch + naive_retriever)': KnowledgeRetrieverConfig(
        retriever_method='mixture',
        retriever_config=MixturedGraphSearchConfig(
            retriever1_name='beamsearch',
            retriever1_config=GraphBeamSearchConfig(),
            retriever2_name='naive_retriever',
            retriever2_config=NaiveGraphSearchConfig()
        ),
        filter_method=None,
        filter_config=None
    )
}

AVAILABLE_QAPIPELINE_CONFIGS = {
    'weak': WeakKGReasonerConfig(),
    'medium': MediumKGReasonerConfig()
}

BASE_KGMODEL_CONFIGS_DIR='./kg_configs'
AVAILABLE_KGMODEL_CONFIGS = {
    'hotpotqa': f"{BASE_KGMODEL_CONFIGS_DIR}/hotpotqa_qwen257b_230126_v2prompts_kgconfig",
    'triviaqa': f"{BASE_KGMODEL_CONFIGS_DIR}/triviaqa_qwen257b_290126_v2prompts_kgconfig",
    'diaasq': f"{BASE_KGMODEL_CONFIGS_DIR}/diaasq_qwen257b_260126_v2prompts_kgconfig",
    'natural_questions': f"{BASE_KGMODEL_CONFIGS_DIR}/naturalqa_qwen257b_020226_v2prompts_kgconfig",
    'musique': f"{BASE_KGMODEL_CONFIGS_DIR}/musique_qwen257b_050226_v2prompts_kgconfig",
    '2wiki': f"{BASE_KGMODEL_CONFIGS_DIR}/2wiki_qwen257b_080226_v2prompts_kgconfig"
}

AVAILABLE_DATASET_QUESTIONS = {
    'hotpotqa': [
        "Were Scott Derrickson and Ed Wood of the same nationality?",
        "What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?",
        "What science fantasy young adult series, told in first person, has a set of companion books narrating the stories of enslaved worlds and alien species?"
    ],
    'triviaqa': [
        "Who had an 80s No 1 hit with Hold On To The Nights?",
        "Men Against the Sea and Pitcairn's Island were two sequels to what famous novel?",
        "Kim Carnes' nine weeks at No 1 with Bette Davis Eyes was interrupted for one week by which song?"
    ],  
    'diaasq': [
        "Which device is better in battery life: iPhone11 Pro Max or Xiaomi 11?",
        "The majority of speakers have positive, neutral or negative sentiment about connection of Apple?",
        "The majority of speakers have positive, neutral or negative sentiment about signal of Apple?"
    ],
    'natural_questions': [
        "who sang what in the world's come over you",
        "who produces the most wool in the world",
        "where does alaska the last frontier take place"
    ],
    'musique': [
        "Who is the spouse of the Green performer?",
        "Who founded the company that distributed the film UHF?",
        "What administrative territorial entity is the owner of Ciudad Deportiva located?"
    ],
    '2wiki': [
        "Who is the mother of the director of film Polish-Russian War (Film)?",
        "Which film came out first, Blind Shaft or The Mask Of Fu Manchu?",
        "When did John V, Prince Of Anhalt-Zerbst's father die?"
    ]
}

#### Start performance testing of QA Pipeline 

In [5]:
SELECTED_DATASET = '2wiki' # TO CHANGE

In [6]:
kg_config = joblib.load(AVAILABLE_KGMODEL_CONFIGS[SELECTED_DATASET])

In [7]:
#kg_config.embedders_configs['default'].device = 'cpu'
kg_config.agents_configs['default'].agent_config.credentials['port'] = 11439

In [8]:
pprint(kg_config)

KnowledgeGraphModelConfig(log_path='log/kg_model/main',
                          verbose=False,
                          log_level=10,
                          graph_struct_config=GraphModelConfig(log_path='log/kg_model/graph',
                                                               verbose=False,
                                                               log_level=10,
                                                               driver_config=GraphDriverConfig(db_vendor='neo4j',
                                                                                               db_config=GraphDBConnectionConfig(db_info={'db': 'personalaigraphdb',
                                                                                                                                          'table': 'personalaigraphtable'},
                                                                                                                                 params={'pwd': 'password',
       

In [9]:
ELAPSED_TIME = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))

In [10]:
for qapipe_version, qapipe_config in AVAILABLE_QAPIPELINE_CONFIGS.items():
    for retriever_name, retriever_config in AVAILABLE_RETRIEVER_CONFIGS.items():
        print("="*20)
        print(f"QA Pipeline / Retriever: {qapipe_version} / {retriever_name}")
        print("="*20)

        qapipe_config.knowledge_retriever_config = retriever_config

        pai_config = PersonalAIConfig(
            lang='en',
            log_level=LogLevel.DISABLED,
            kg_model_config=kg_config,
            qa_pipeline_config=QAPipelineConfig(
                preprocessor_config=QueryPreprocessorConfig(
                    denoising_config=None,
                    enhancing_config=None,
                    decomposition_config=None
                ),
                reasoner_config=KnowledgeGraphReasonerConfig(
                    reasoner_name=qapipe_version,
                    reasoner_config=qapipe_config
                )
            )
        )

        for log_level in [LogLevel.DEBUG, LogLevel.DISABLED]:
            pai_config.log_level = log_level
            personalai = PersonalAI(pai_config, cache_kvdriver_config=None)

            print("-"*20)
            print("KG model info:\n", personalai.kg_model.count_items())
            print("Log Level: ", log_level)
            print("-"*20)

            for question in AVAILABLE_DATASET_QUESTIONS[SELECTED_DATASET]:
                print("Question: ", question)
                
                s_time = time()
                _, _, _ = personalai.answer_question(question)
                e_time = time()
                cur_elapsed_time = round(e_time-s_time, 5)
                print("Elapsed time: ", cur_elapsed_time)
                print()

                ELAPSED_TIME[qapipe_version][log_level][retriever_name].append(cur_elapsed_time)


            personalai.close_connections()
            del personalai

QA Pipeline / Retriever: weak / naive_retriever
--------------------
KG model info:
 {'graph_info': {'triplets': 301483, 'nodes': 94425}, 'embeddings_info': {'nodes': 94425, 'triplets': 84163}, 'nodestree_info': None}
Log Level:  10
--------------------
Question:  Who is the mother of the director of film Polish-Russian War (Film)?
Elapsed time:  5.08734

Question:  Which film came out first, Blind Shaft or The Mask Of Fu Manchu?
Elapsed time:  6.33682

Question:  When did John V, Prince Of Anhalt-Zerbst's father die?
Elapsed time:  4.18389

--------------------
KG model info:
 {'graph_info': {'triplets': 301483, 'nodes': 94425}, 'embeddings_info': {'nodes': 94425, 'triplets': 84163}, 'nodestree_info': None}
Log Level:  51
--------------------
Question:  Who is the mother of the director of film Polish-Russian War (Film)?
Elapsed time:  4.39811

Question:  Which film came out first, Blind Shaft or The Mask Of Fu Manchu?
Elapsed time:  4.79548

Question:  When did John V, Prince Of Anha

'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: 2f222f9c-af39-40f3-9815-318c4e273c30)')' thrown while requesting HEAD https://huggingface.co/intfloat/multilingual-e5-large/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].


--------------------
KG model info:
 {'graph_info': {'triplets': 301483, 'nodes': 94425}, 'embeddings_info': {'nodes': 94425, 'triplets': 84163}, 'nodestree_info': None}
Log Level:  51
--------------------
Question:  Who is the mother of the director of film Polish-Russian War (Film)?
Elapsed time:  199.50656

Question:  Which film came out first, Blind Shaft or The Mask Of Fu Manchu?
Elapsed time:  131.5709

Question:  When did John V, Prince Of Anhalt-Zerbst's father die?
Elapsed time:  159.8548



'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: 6ce26a16-23d6-4603-ad9c-4a14081df708)')' thrown while requesting HEAD https://huggingface.co/intfloat/multilingual-e5-large/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].


QA Pipeline / Retriever: medium / watercircles
--------------------
KG model info:
 {'graph_info': {'triplets': 301483, 'nodes': 94425}, 'embeddings_info': {'nodes': 94425, 'triplets': 84163}, 'nodestree_info': None}
Log Level:  10
--------------------
Question:  Who is the mother of the director of film Polish-Russian War (Film)?
Elapsed time:  122.3851

Question:  Which film came out first, Blind Shaft or The Mask Of Fu Manchu?
Elapsed time:  28.61768

Question:  When did John V, Prince Of Anhalt-Zerbst's father die?
Elapsed time:  22.06558



...

--------------------
KG model info:
 {'graph_info': {'triplets': 301483, 'nodes': 94425}, 'embeddings_info': {'nodes': 94425, 'triplets': 84163}, 'nodestree_info': None}
Log Level:  51
--------------------
Question:  Who is the mother of the director of film Polish-Russian War (Film)?
Elapsed time:  265.80225

Question:  Which film came out first, Blind Shaft or The Mask Of Fu Manchu?
Elapsed time:  80.32714

Question:  When did John V, Prince Of Anhalt-Zerbst's father die?
Elapsed time:  172.43742



'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: 71e7c7fa-d985-4768-9190-d7f57d94051a)')' thrown while requesting HEAD https://huggingface.co/intfloat/multilingual-e5-large/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].


QA Pipeline / Retriever: medium / mixture (beamsearch + naive_retriever)
--------------------
KG model info:
 {'graph_info': {'triplets': 301483, 'nodes': 94425}, 'embeddings_info': {'nodes': 94425, 'triplets': 84163}, 'nodestree_info': None}
Log Level:  10
--------------------
Question:  Who is the mother of the director of film Polish-Russian War (Film)?
Elapsed time:  65.35983

Question:  Which film came out first, Blind Shaft or The Mask Of Fu Manchu?
Elapsed time:  214.7924

Question:  When did John V, Prince Of Anhalt-Zerbst's father die?
Elapsed time:  238.96354



'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: b74e8fcd-760b-4d2d-86a1-c9d3fa7c33e0)')' thrown while requesting HEAD https://huggingface.co/intfloat/multilingual-e5-large/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].


--------------------
KG model info:
 {'graph_info': {'triplets': 301483, 'nodes': 94425}, 'embeddings_info': {'nodes': 94425, 'triplets': 84163}, 'nodestree_info': None}
Log Level:  51
--------------------
Question:  Who is the mother of the director of film Polish-Russian War (Film)?
Elapsed time:  64.6795

Question:  Which film came out first, Blind Shaft or The Mask Of Fu Manchu?
Elapsed time:  215.24151

Question:  When did John V, Prince Of Anhalt-Zerbst's father die?
Elapsed time:  240.1639



In [12]:
pprint(dict(ELAPSED_TIME))

{'medium': defaultdict(<function <lambda>.<locals>.<lambda> at 0x7f47ac702710>,
                       {10: defaultdict(<class 'list'>,
                                        {'beamsearch': [190.59973,
                                                        132.77238,
                                                        160.52285],
                                         'mixture (beamsearch + naive_retriever)': [65.35983,
                                                                                    214.7924,
                                                                                    238.96354],
                                         'mixture (beamsearch + watercircles)': [272.64711,
                                                                                 86.16372,
                                                                                 174.72731],
                                         'naive_retriever': [23.05306,
                               

In [14]:
for k, v in ELAPSED_TIME.items():
    print(k)
    for vk, vv in v.items():
        print(vk)
        for vvk, vvv in vv.items():
            print(vvk, vvv)

weak
10
naive_retriever [5.08734, 6.33682, 4.18389]
beamsearch [31.54336, 22.43662, 28.26874]
watercircles [8.75818, 5.65208, 6.99157]
mixture (beamsearch + watercircles) [31.96782, 23.96483, 32.46391]
mixture (beamsearch + naive_retriever) [27.58769, 21.72227, 27.80472]
51
naive_retriever [4.39811, 4.79548, 4.33383]
beamsearch [28.51001, 21.9846, 29.28085]
watercircles [8.56139, 5.63405, 6.96282]
mixture (beamsearch + watercircles) [32.44134, 23.11175, 33.31866]
mixture (beamsearch + naive_retriever) [29.05589, 22.70446, 28.39558]
medium
10
naive_retriever [23.05306, 55.00223, 57.85489]
beamsearch [190.59973, 132.77238, 160.52285]
watercircles [122.3851, 28.61768, 22.06558]
mixture (beamsearch + watercircles) [272.64711, 86.16372, 174.72731]
mixture (beamsearch + naive_retriever) [65.35983, 214.7924, 238.96354]
51
naive_retriever [21.88607, 55.43624, 57.43457]
beamsearch [199.50656, 131.5709, 159.8548]
watercircles [130.15406, 40.33089, 23.45323]
mixture (beamsearch + watercircles) [2

#### Performance testing results (log)

-----------------------------

#### <b>22.03.26 | version=2.3.1</b>

##### <b>hotpotqa</b>

ssh port forwarding params:

-L 7503:localhost:7503 -L 7716:localhost:7716 -L 6337:localhost:6337 -L 6438:localhost:6438 -L 27046:localhost:27046 -L 8110:localhost:8110 -L 6408:localhost:6408 -L 5569:localhost:5569 -L 9212:localhost:9212 -L 9610:localhost:9610

results:

##### <b>triviaqa</b>

ssh port forwarding params:

-L 7507:localhost:7507 -L 7720:localhost:7720 -L 6341:localhost:6341 -L 6442:localhost:6442 -L 27050:localhost:27050 -L 8114:localhost:8114 -L 6412:localhost:6412 -L 5573:localhost:5573 -L 9216:localhost:9216 -L 9614:localhost:9614

results:


##### <b>diaasq</b>

ssh port forwarding params:

-L 7511:localhost:7511 -L 7724:localhost:7724 -L 6345:localhost:6345 -L 6446:localhost:6446 -L 27054:localhost:27054 -L 8118:localhost:8118 -L 6416:localhost:6416 -L 5577:localhost:5577 -L 9220:localhost:9220 -L 9618:localhost:9618

results:


##### <b>naturalqa</b>

ssh port forwarding params:

-L 7515:localhost:7515 -L 7728:localhost:7728 -L 6349:localhost:6349 -L 6450:localhost:6450 -L 27058:localhost:27058 -L 8122:localhost:8122 -L 6420:localhost:6420 -L 5581:localhost:5581 -L 9224:localhost:9224 -L 9622:localhost:9622

results:


##### <b>musique</b>

ssh port forwarding params:

-L 7519:localhost:7519 -L 7732:localhost:7732 -L 6353:localhost:6353 -L 6454:localhost:6454 -L 27062:localhost:27062 -L 8126:localhost:8126 -L 6424:localhost:6424 -L 5585:localhost:5585 -L 9228:localhost:9228 -L 9626:localhost:9626

results:


##### <b>2wiki</b>

ssh port forwarding params:

-L 7523:localhost:7523 -L 7736:localhost:7736 -L 6357:localhost:6357 -L 6458:localhost:6458 -L 27066:localhost:27066 -L 8130:localhost:8130 -L 6428:localhost:6428 -L 5589:localhost:5589 -L 9232:localhost:9232 -L 9630:localhost:9630

results:

* weak
    * Log Enbaled
        * naive_retriever [5.08734, 6.33682, 4.18389]
        * beamsearch [31.54336, 22.43662, 28.26874]
        * watercircles [8.75818, 5.65208, 6.99157]
        * mixture (beamsearch + watercircles) [31.96782, 23.96483, 32.46391]
        * mixture (beamsearch + naive_retriever) [27.58769, 21.72227, 27.80472]
    * Log Disabled
        * naive_retriever [4.39811, 4.79548, 4.33383]
        * beamsearch [28.51001, 21.9846, 29.28085]
        * watercircles [8.56139, 5.63405, 6.96282]
        * mixture (beamsearch + watercircles) [32.44134, 23.11175, 33.31866]
        * mixture (beamsearch + naive_retriever) [29.05589, 22.70446, 28.39558]
* medium
    * Log Enbaled
        * naive_retriever [23.05306, 55.00223, 57.85489]
        * beamsearch [190.59973, 132.77238, 160.52285]
        * watercircles [122.3851, 28.61768, 22.06558]
        * mixture (beamsearch + watercircles) [272.64711, 86.16372, 174.72731]
        * mixture (beamsearch + naive_retriever) [65.35983, 214.7924, 238.96354]
    * Log Disabled
        * naive_retriever [21.88607, 55.43624, 57.43457]
        * beamsearch [199.50656, 131.5709, 159.8548]
        * watercircles [130.15406, 40.33089, 23.45323]
        * mixture (beamsearch + watercircles) [265.80225, 80.32714, 172.43742]
        * mixture (beamsearch + naive_retriever) [64.6795, 215.24151, 240.1639]


-----------------------------